# EXERCISE 4

In a shop floor, the head of the quality assurance department is interested in keeping under control the stability of a turning process.
Every day, the cylindrical rings are produced in four temporally consecutive batches denoted as A (early morning), B (late morning), C (early afternoon), D (late afternoon).
One cylindrical ring is collected and its outer diameter (cm) is measured every day in each batch.
A dataset consisting of 25 consecutive sample collections is stored in 'diameters.csv'.
Identify a suitable model.

In [ ]:
# Import the necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import seaborn as sns
import qdatoolkit as qda

# Import the dataset
data = pd.read_csv('../Data/diameters.csv')

# Inspect the dataset
data.head()

In [ ]:
# Plot the data 
plt.plot(data['Diameter'], 'o-')
plt.xlabel('Index')
plt.ylabel('measuremet')
plt.title('Time series plot of Diameter')
plt.grid()
plt.show()

In [ ]:
_ = qda.Assumptions(data['Diameter']).independence()

> Runs test would lead to not reject the randmness assumption at 95% confidence, but the time series exhibits a clear pattern.

> <t1 style="color:red"> Batch effect? </t1>

> Let's create DAY and BATCH variables

In [ ]:
# Create Batch variable
data['Batch'] = np.tile(np.arange(1, 5), int(len(data)/4)) #tile the simple batches array ([1 2 3 4]) for 25 times
data['Day'] = np.repeat(np.arange(1, len(data)/4+1), 4) #repeate the element of the days array [from 1 to 25] for time each element

In [ ]:
# Plot the data as 4 separate batches
#we need to extract the data for each batch; change the color, the line, and the marker; assign a label for each batch

plt.plot(data['Diameter'][data['Batch'] == 1], 'o:b', label = 'Batch 1') 
plt.plot(data['Diameter'][data['Batch'] == 2], 's:r', label = 'Batch 2')
plt.plot(data['Diameter'][data['Batch'] == 3], 'D:g', label = 'Batch 3')
plt.plot(data['Diameter'][data['Batch'] == 4], '^:m', label = 'Batch 4')

plt.xlabel('Index')
plt.ylabel('Diameter')
plt.legend()
plt.title('Time series plot of Diameter')
plt.grid()
plt.show()

> <t1 style="color:red"> Batch 3 yields systematically a larger diameter than the other batches. </t1>

In [ ]:
#Other possible graphs: scatterplot of Diameter VS batch
plt.scatter(data['Batch'], data['Diameter'])
plt.xlabel('Batch')
plt.ylabel('Diameter')
plt.title('Scatterplot of Diameter VS Batch')
plt.show()

In [ ]:
#Other possible graphs: scatterplot of Diameter VS day
plt.scatter(data['Day'], data['Diameter'])
plt.xlabel('Day')
plt.ylabel('Diameter')
plt.title('Scatterplot of Diameter VS Day')
plt.show()

> # Which type of model?  
> Dummy variable:
> - $=0$, for batches A, B and D
> - $=1$, for batch C

In [ ]:
# create the dummy variable 
data['Dummy'] = np.tile(np.array([0, 0, 1, 0]), int(len(data)/4))


In [ ]:
#calculate a regression model with constant and dummy
import statsmodels.api as sm
import qdatoolkit as qda

x = data['Dummy']
x = sm.add_constant(data['Dummy']) 
y = data['Diameter']
model = sm.OLS(y, x).fit()

qda.summary(model)

In [ ]:
fig, axs = plt.subplots(2, 2)
fig.suptitle('Residual Plots')
stats.probplot(model.resid, dist="norm", plot=axs[0,0])
axs[0,0].set_title('Normal probability plot')
axs[0,1].scatter(model.fittedvalues, model.resid)
axs[0,1].set_title('Versus Fits')
fig.subplots_adjust(hspace=0.5)
axs[1,0].hist(model.resid)
axs[1,0].set_title('Histogram')
axs[1,1].plot(np.arange(1, len(model.resid)+1), model.resid, 'o-')
plt.show()

In [ ]:
_ = qda.Assumptions(model.resid).normality()

In [ ]:
_ = qda.Assumptions(model.resid).independence()

> The model is adequate: Residuals and normal and independent.

> What if we used a different dummy variable definition?  
> Let's try to use the Batch variable as a set of categorical dummy variables.

In [ ]:
#create a vectors for dummy variables associated to each batch
data['Dummy_Batch1'] = np.where(data['Batch']==1, 1, 0)
data['Dummy_Batch2'] = np.where(data['Batch']==2, 1, 0)
data['Dummy_Batch3'] = np.where(data['Batch']==3, 1, 0)
data['Dummy_Batch4'] = np.where(data['Batch']==4, 1, 0)


In [ ]:
x = data[['Dummy_Batch1', 'Dummy_Batch2', 'Dummy_Batch3', 'Dummy_Batch4']]
x = sm.add_constant(x) 
y = data['Diameter']
model = sm.OLS(y, x).fit()

qda.summary(model)

In [ ]:
fig, axs = plt.subplots(2, 2)
fig.suptitle('Residual Plots')
stats.probplot(model.resid, dist="norm", plot=axs[0,0])
axs[0,0].set_title('Normal probability plot')
axs[0,1].scatter(model.fittedvalues, model.resid)
axs[0,1].set_title('Versus Fits')
fig.subplots_adjust(hspace=0.5)
axs[1,0].hist(model.resid)
axs[1,0].set_title('Histogram')
axs[1,1].plot(np.arange(1, len(model.resid)+1), model.resid, 'o-')
plt.show()

In [ ]:
_ = qda.Assumptions(model.resid).normality()

In [ ]:
_ = qda.Assumptions(model.resid).independence()

> The model is similar, with an R-squared adjusted of 0.599 (against the previous of 0.603).